# Stylométrie 
Faire des recherche de distance avec les quatre types de distance:
* PNG

In [3]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

# ==============================
# PARAMÈTRES À ADAPTER
# ==============================

# Dossier contenant les fichiers .txt
corpus_dir = "/Users/elodiepaupe/Documents/Corpus22_MGH/lemma_lasla_txt_propre"

# Nombre de mots les plus fréquents à conserver (MFW)
n_mfw = 500

# Largeur voulue par étiquette dans le PNG
mm_par_titre = 4

# Résolution de sortie
png_dpi = 300

# Hauteur de la figure en pouces
height_inch = 10

# Taille de police des étiquettes
leaf_font_size = 6

# Méthode de linkage pour le dendrogramme
linkage_method = "average"

# Dossier de sortie
output_dir = os.path.join(corpus_dir, "dendrogrammes_png")
os.makedirs(output_dir, exist_ok=True)


# ==============================
# FONCTIONS
# ==============================

def read_file(filepath):
    """Lit un fichier texte en UTF-8."""
    with open(filepath, "r", encoding="utf-8") as file:
        return file.read()


def clean_label(filename):
    """Prépare un label plus lisible à partir du nom de fichier."""
    label = os.path.splitext(filename)[0]
    label = re.sub(r"[_]+", " ", label)
    return label


def burrows_delta_matrix(df_freq):
    """
    Calcule la distance Delta de Burrows.
    1. Transformation des fréquences relatives en z-scores par mot.
    2. Distance = moyenne des écarts absolus entre z-scores.
    """
    means = df_freq.mean(axis=0)
    stds = df_freq.std(axis=0, ddof=0).replace(0, np.nan)

    z_scores = (df_freq - means) / stds
    z_scores = z_scores.fillna(0)

    return pdist(z_scores.values, metric="cityblock") / z_scores.shape[1]


def figure_width_from_labels(n_labels, mm_per_label=4):
    """Convertit une largeur de mm par label en largeur matplotlib en pouces."""
    width_mm = n_labels * mm_per_label
    return width_mm / 25.4


def save_dendrogram(linkage_matrix, labels, distance_name, output_dir):
    """Génère et sauvegarde un dendrogramme PNG avec largeur dynamique."""
    n_texts = len(labels)
    width_inch = figure_width_from_labels(n_texts, mm_par_titre)

    plt.figure(figsize=(width_inch, height_inch))

    dendrogram(
        linkage_matrix,
        labels=labels,
        leaf_rotation=90,
        leaf_font_size=leaf_font_size
    )
    

    plt.title(f"Dendrogramme - Distance : {distance_name}")
    plt.xlabel("Textes")
    plt.ylabel("Distance")
    plt.tight_layout()

    output_path = os.path.join(output_dir, f"dendrogramme_{distance_name}.png")
    plt.savefig(output_path, format="png", dpi=png_dpi, bbox_inches="tight")
    plt.close()

    print(f"Image sauvegardée : {output_path}")


# ==============================
# CHARGEMENT DU CORPUS
# ==============================

files = sorted([f for f in os.listdir(corpus_dir) if f.endswith(".txt")])

if not files:
    raise FileNotFoundError(f"Aucun fichier .txt trouvé dans : {corpus_dir}")

texts = [read_file(os.path.join(corpus_dir, file)) for file in files]
labels = [clean_label(file) for file in files]


# ==============================
# MATRICE MFW
# ==============================

# En stylométrie, on évite généralement de supprimer les stopwords,
# car les mots-outils sont souvent des marqueurs stylistiques importants.
vectorizer = CountVectorizer(
    max_features=n_mfw,
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b"
)

X_counts = vectorizer.fit_transform(texts).toarray()

# Lignes = textes ; colonnes = MFW
df_counts = pd.DataFrame(
    X_counts,
    index=files,
    columns=vectorizer.get_feature_names_out()
)

# Fréquences relatives pour neutraliser les différences de longueur des textes
df_freq = df_counts.div(df_counts.sum(axis=1), axis=0).fillna(0)

print(f"Nombre de textes : {len(files)}")
print(f"Nombre de MFW utilisés : {df_freq.shape[1]}")
print(f"Largeur PNG : {mm_par_titre} mm par titre")


# ==============================
# DISTANCES À CALCULER
# ==============================

distance_metrics = {
    "Euclidean": "euclidean",
    "Manhattan": "cityblock",
    "Cosine": "cosine",
    "Chebyshev": "chebyshev",
    "Delta": "burrows_delta"
}


# ==============================
# CALCUL ET EXPORT PNG
# ==============================

for distance_name, metric in distance_metrics.items():

    if metric == "burrows_delta":
        dist_matrix = burrows_delta_matrix(df_freq)
    else:
        dist_matrix = pdist(df_freq.values, metric=metric)

    linkage_matrix = linkage(dist_matrix, method=linkage_method)

    save_dendrogram(
        linkage_matrix=linkage_matrix,
        labels=labels,
        distance_name=distance_name,
        output_dir=output_dir
    )


Nombre de textes : 416
Nombre de MFW utilisés : 500
Largeur PNG : 4 mm par titre
Image sauvegardée : /Users/elodiepaupe/Documents/Corpus22_MGH/lemma_lasla_txt_propre/dendrogrammes_png/dendrogramme_Euclidean.png
Image sauvegardée : /Users/elodiepaupe/Documents/Corpus22_MGH/lemma_lasla_txt_propre/dendrogrammes_png/dendrogramme_Manhattan.png
Image sauvegardée : /Users/elodiepaupe/Documents/Corpus22_MGH/lemma_lasla_txt_propre/dendrogrammes_png/dendrogramme_Cosine.png
Image sauvegardée : /Users/elodiepaupe/Documents/Corpus22_MGH/lemma_lasla_txt_propre/dendrogrammes_png/dendrogramme_Chebyshev.png
Image sauvegardée : /Users/elodiepaupe/Documents/Corpus22_MGH/lemma_lasla_txt_propre/dendrogrammes_png/dendrogramme_Delta.png
